# NB61: Real-time AdTech Bidding

Kafka -> Spark -> Redis/Cassandra

## 1. Environment Setup

Installs **Java 8**, **Spark 3.5.0**, **Kafka 3.6.1**, and Python libraries (PySpark, Kafka-Python, Redis, Mongo, ES, Cassandra, MinIO).

In [ ]:
# Install Dependencies (Java 8, Spark 3.5.0, Kafka 3.6.1)
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!wget -q https://archive.apache.org/dist/kafka/3.6.1/kafka_2.13-3.6.1.tgz
!tar xf kafka_2.13-3.6.1.tgz
!pip install -q "numpy<2.0.0" findspark pyspark kafka-python redis pymongo elasticsearch==7.10.1 cassandra-driver minio

# Environment Variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"
import findspark
findspark.init()

## 2. Start Services

Starts background services needed for this pipeline:
- **Kafka** (Zookeeper + Broker)
- **Redis**
- **Cassandra**

In [ ]:
# Start Kafka
!./kafka_2.13-3.6.1/bin/zookeeper-server-start.sh -daemon ./kafka_2.13-3.6.1/config/zookeeper.properties
!./kafka_2.13-3.6.1/bin/kafka-server-start.sh -daemon ./kafka_2.13-3.6.1/config/server.properties
# Start Redis
!apt-get install redis-server -qq > /dev/null
!service redis-server start
# Start Cassandra
!wget -q https://archive.apache.org/dist/cassandra/4.1.3/apache-cassandra-4.1.3-bin.tar.gz
!tar xf apache-cassandra-4.1.3-bin.tar.gz
!apache-cassandra-4.1.3/bin/cassandra -R > cassandra.log 2>&1 &

import time, socket, os
def wait_for_port(port, host='localhost', timeout=120):
    start_time = time.time()
    while True:
        try:
            with socket.create_connection((host, port), timeout=1):
                print(f"Service at {host}:{port} is ready!")
                return True
        except (OSError, ConnectionRefusedError):
            if time.time() - start_time > timeout:
                print(f"Timeout waiting for {host}:{port} to start.")
                # Dump logs for debugging
                if os.path.exists('minio.log'):
                    print('--- MINIO LOG ---')
                    print(open('minio.log').read())
                if os.path.exists('es.log'):
                    print('--- ES LOG ---')
                    print(open('es.log').read())
                if os.path.exists('cassandra.log'):
                    print('--- CASSANDRA LOG ---')
                    print(open('cassandra.log').read())
                raise Exception(f"Service at {host}:{port} failed to start.")
            time.sleep(2)

# Wait for services
wait_for_port(9092) # Kafka
wait_for_port(9042) # Cassandra
time.sleep(10) # Extra buffer for Cassandra
wait_for_port(6379) # Redis


## 3. Create Kafka Topic

Creates a topic named `input-topic`.

In [ ]:
# Create Topic
!./kafka_2.13-3.6.1/bin/kafka-topics.sh --create --topic input-topic --bootstrap-server localhost:9092 --replication-factor 1 --partitions 1

## 4. Producer (Bid Simulator)

Simulates bid requests with `bid_id`, `user_id`, `site`.

In [ ]:
import threading
import time, json, random
from kafka import KafkaProducer

def send_data():
    producer = None
    # Retry connection
    while not producer:
        try:
            producer = KafkaProducer(bootstrap_servers='localhost:9092')
        except Exception as e:
            print(f"Waiting for Kafka... {e}")
            time.sleep(2)
    
    # Send Loop
    while True:
        try:
            bid = {'bid_id': f'b{random.randint(1000,99999)}', 'user_id': f'u{random.randint(1,10)}', 'site': 'example.com', 'bid_floor': random.uniform(0.1, 1.0)}
            producer.send('input-topic', json.dumps(bid).encode('utf-8'))
            time.sleep(0.1)
        except Exception as e:
            print(f"Producer Error: {e}")
            time.sleep(1)

print("Starting Bid Simulator (Background Thread)...")
t = threading.Thread(target=send_data)
t.daemon = True
t.start()
print("Producer running continuously...")

## 5. Bid Decision Engine (Spark)

1. Reads Bids from Kafka.
2. Looks up User Profile in Redis (`standard` vs `premium`).
3. Makes a BID/PASS decision.
4. Logs the decision to Cassandra (`adtech.bids`).

In [ ]:
%%writefile kafka_consumer.py
from pyspark.sql import SparkSession
import json
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, FloatType
import redis
from cassandra.cluster import Cluster

# Setup Redis & Cassandra
r = redis.Redis()
for i in range(1, 11): r.set(f"u{i}", "premium" if i % 2 == 0 else "standard")

cluster = Cluster(['127.0.0.1'])
session = cluster.connect()
session.execute("CREATE KEYSPACE IF NOT EXISTS adtech WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1}")
session.execute("CREATE TABLE IF NOT EXISTS adtech.bids (bid_id text PRIMARY KEY, user_type text, action text)")
session.shutdown()

spark = SparkSession.builder.appName("AdTech").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, epoch_id):
    rows = df.collect()
    if not rows: return
    r_local = redis.Redis()
    cluster_local = Cluster(['127.0.0.1'])
    session_local = cluster_local.connect('adtech')
    for row in rows:
        data = json.loads(row.value)
        bid_id = data['bid_id']
        user_id = data['user_id']
        # Lookup
        u_type = r_local.get(user_id)
        u_type = u_type.decode('utf-8') if u_type else 'unknown'
        # Decision
        action = "BID" if u_type == "premium" else "PASS"
        # Log
        session_local.execute(f"INSERT INTO bids (bid_id, user_type, action) VALUES ('{bid_id}', '{u_type}', '{action}')")
    session_local.shutdown()
    print(f"Batch {epoch_id} processed {len(rows)} logic. Decisions logged to Cassandra.")

print("Starting Spark Streaming Job...")
df = spark.readStream.format("kafka").option("kafka.bootstrap.servers", "localhost:9092").option("subscribe", "input-topic").option("startingOffsets", "earliest").load()
query = df.selectExpr("CAST(value AS STRING)").writeStream.foreachBatch(process_batch).start()
query.awaitTermination(30)
print("Spark Job Finished.")

In [ ]:
!spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 kafka_consumer.py

## 6. Verification

Query Cassandra for bid logs.

In [ ]:
from cassandra.cluster import Cluster
import time

print("Verifying Cassandra data...")
cluster = Cluster(['127.0.0.1'])
session = cluster.connect('adtech')

# Retry logic for eventual consistency
for _ in range(5):
    rows = list(session.execute("SELECT * FROM bids LIMIT 10"))
    if rows: break
    time.sleep(2)

print(f"--- Found {len(rows)} Ad Bids ---")
for row in rows: print(row)

session.shutdown()
assert len(rows) > 0, "Verification Failed: No data found in Cassandra!"